# Pit Stop Prediction OOP Solution

A readable object-oriented version of the original notebook.

In [ ]:
"""Import libraries, define settings, and make the notebook reproducible."""

import os
import gc
import random
import warnings
from dataclasses import dataclass, field

import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.float_format", lambda x: f"{x:.6f}")


@dataclass
class Config:
    seed: int = 42
    target: str = "PitNextLap"
    id_col: str = "id"
    n_folds: int = 5
    comp_paths: list[str] = field(
        default_factory=lambda: [
            "/kaggle/input/competitions/playground-series-s6e5",
            "/kaggle/input/playground-series-s6e5",
        ]
    )
    original_paths: list[str] = field(
        default_factory=lambda: [
            "/kaggle/input/f1-strategy-dataset-pit-stop-prediction/f1_strategy_dataset_v4.csv",
            "/kaggle/input/datasets/aadigupta1601/f1-strategy-dataset-pit-stop-prediction/f1_strategy_dataset_v4.csv",
        ]
    )
    numeric_features: list[str] = field(
        default_factory=lambda: [
            "Year",
            "PitStop",
            "LapNumber",
            "Stint",
            "TyreLife",
            "Position",
            "LapTime (s)",
            "LapTime_Delta",
            "Cumulative_Degradation",
            "RaceProgress",
            "Position_Change",
        ]
    )
    categorical_features: list[str] = field(default_factory=lambda: ["Driver", "Compound", "Race"])
    base_cat_cols: list[str] = field(default_factory=lambda: ["Driver", "Compound", "Race"])


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)


def first_existing_path(paths: list[str]) -> str:
    for path in paths:
        if os.path.exists(path):
            return path
    raise FileNotFoundError(f"No valid path found from: {paths}")


cfg = Config()
seed_everything(cfg.seed)


## Data Loading and Checks

In [ ]:
"""Load train, test, submission, and optional original data, then show compact dataset checks."""

class DataLoader:
    def __init__(self, cfg: Config):
        self.cfg = cfg

    def load(self):
        comp_path = first_existing_path(self.cfg.comp_paths)
        train = pd.read_csv(os.path.join(comp_path, "train.csv"))
        test = pd.read_csv(os.path.join(comp_path, "test.csv"))
        sample_submission = pd.read_csv(os.path.join(comp_path, "sample_submission.csv"))
        original = self._load_original()
        return train, test, sample_submission, original

    def _load_original(self):
        for path in self.cfg.original_paths:
            if os.path.exists(path):
                return pd.read_csv(path)
        return None

    def dataset_overview(self, train, test, original):
        rows = [
            self._one_dataset_row("train", train),
            self._one_dataset_row("test", test),
        ]
        if original is not None:
            rows.append(self._one_dataset_row("original", original))
        return pd.DataFrame(rows)

    def _one_dataset_row(self, name, df):
        target_rate = np.nan
        if self.cfg.target in df.columns:
            target_rate = df[self.cfg.target].mean()
        id_duplicates = np.nan
        if self.cfg.id_col in df.columns:
            id_duplicates = df[self.cfg.id_col].duplicated().sum()
        return {
            "dataset": name,
            "rows": len(df),
            "columns": df.shape[1],
            "target_rate": target_rate,
            "missing_values": int(df.isna().sum().sum()),
            "duplicate_ids": id_duplicates,
            "duplicate_rows": int(df.duplicated().sum()),
        }

    def schema_summary(self, df, name):
        return pd.DataFrame(
            {
                "dataset": name,
                "column": df.columns,
                "dtype": [str(df[col].dtype) for col in df.columns],
                "missing": [int(df[col].isna().sum()) for col in df.columns],
                "missing_pct": [100 * df[col].isna().mean() for col in df.columns],
                "unique": [df[col].nunique() for col in df.columns],
            }
        ).sort_values(["missing_pct", "unique"], ascending=[False, False])


loader = DataLoader(cfg)
train_raw, test_raw, sample_submission, original_raw = loader.load()

display(loader.dataset_overview(train_raw, test_raw, original_raw))
display(loader.schema_summary(train_raw, "train").head(20))

if original_raw is not None:
    display(loader.schema_summary(original_raw, "original").head(20))

gc.collect()


## Quick Drift View

In [ ]:
"""Compare important train and test columns with small drift and coverage tables."""

class DataInspector:
    def __init__(self, cfg: Config):
        self.cfg = cfg

    def numeric_drift(self, train_df, test_df):
        rows = []
        for col in self.cfg.numeric_features:
            if col not in train_df.columns or col not in test_df.columns:
                continue
            train_values = train_df[col].astype(float)
            test_values = test_df[col].astype(float)
            pooled_std = np.sqrt((train_values.var() + test_values.var()) / 2.0)
            smd = 0.0 if pooled_std == 0 else (test_values.mean() - train_values.mean()) / pooled_std
            rows.append(
                {
                    "feature": col,
                    "train_mean": train_values.mean(),
                    "test_mean": test_values.mean(),
                    "smd": smd,
                }
            )
        return pd.DataFrame(rows).sort_values("smd", key=np.abs, ascending=False)

    def categorical_coverage(self, train_df, test_df):
        rows = []
        for col in self.cfg.categorical_features:
            if col not in train_df.columns or col not in test_df.columns:
                continue
            train_values = set(train_df[col].astype("string").dropna().unique())
            test_values = set(test_df[col].astype("string").dropna().unique())
            rows.append(
                {
                    "feature": col,
                    "train_unique": len(train_values),
                    "test_unique": len(test_values),
                    "unseen_in_test": len(test_values - train_values),
                }
            )
        return pd.DataFrame(rows)


inspector = DataInspector(cfg)
display(inspector.numeric_drift(train_raw, test_raw))
display(inspector.categorical_coverage(train_raw, test_raw))


## Feature Builder

In [ ]:
"""Create domain features, align all datasets, and fill missing values for CatBoost."""

class FeatureEngineer:
    def __init__(self, cfg: Config):
        self.cfg = cfg
        self.cat_cols: list[str] = []
        self.num_cols: list[str] = []
        self.feature_cols: list[str] = []

    @staticmethod
    def safe_div(a, b, eps=1e-6):
        return a / (b + eps)

    def add_features(self, df: pd.DataFrame) -> pd.DataFrame:
        out = df.copy()
        eps = 1e-6

        for col in self.cfg.base_cat_cols:
            if col in out.columns:
                out[col] = out[col].astype("string").fillna("__MISSING__").astype(str)

        def has(cols):
            return set(cols).issubset(out.columns)

        if has(["LapNumber", "RaceProgress"]):
            race_progress = out["RaceProgress"].clip(lower=eps)
            est_total = self.safe_div(out["LapNumber"], race_progress, eps).replace([np.inf, -np.inf], np.nan)
            out["EstimatedTotalLaps"] = est_total.clip(1, 120)
            out["LapsRemaining"] = (out["EstimatedTotalLaps"] - out["LapNumber"]).clip(lower=0)
            out["RemainingRaceProgress"] = 1.0 - out["RaceProgress"]
            out["LapProgress_x_LapNumber"] = out["LapNumber"] * out["RaceProgress"]
            out["RacePhase"] = pd.cut(
                out["RaceProgress"],
                bins=[-np.inf, 0.20, 0.40, 0.60, 0.80, np.inf],
                labels=["P1", "P2", "P3", "P4", "P5"],
            ).astype(str)
            out["LapBin"] = pd.cut(
                out["LapNumber"],
                bins=[-np.inf, 5, 10, 20, 35, 50, np.inf],
                labels=["L_000_005", "L_006_010", "L_011_020", "L_021_035", "L_036_050", "L_051_plus"],
            ).astype(str)

        if has(["TyreLife", "LapNumber"]):
            out["TyreAgeRatio"] = self.safe_div(out["TyreLife"], out["LapNumber"].clip(lower=1), eps)
            out["LapPerTyreLife"] = self.safe_div(out["LapNumber"], out["TyreLife"] + 1, eps)
            out["TyreLifeMinusLap"] = out["TyreLife"] - out["LapNumber"]

        if has(["TyreLife", "EstimatedTotalLaps"]):
            out["TyreAgeVsRace"] = self.safe_div(out["TyreLife"], out["EstimatedTotalLaps"].clip(lower=1), eps)

        if has(["TyreLife", "RaceProgress"]):
            out["PitWindowPressure"] = out["TyreLife"] * out["RaceProgress"]

        if has(["Stint", "TyreLife"]):
            out["StintPressure"] = out["Stint"] * out["TyreLife"]
            out["TyreLife_x_Stint"] = out["TyreLife"] * out["Stint"]
            out["Is_First_Stint"] = (out["Stint"] == 1).astype(np.int8)
            out["Is_Late_Stint"] = (out["Stint"] >= 3).astype(np.int8)

        if "TyreLife" in out.columns:
            out["TyreLifeBin"] = pd.cut(
                out["TyreLife"],
                bins=[-np.inf, 3, 7, 12, 20, 30, np.inf],
                labels=["T_000_003", "T_004_007", "T_008_012", "T_013_020", "T_021_030", "T_031_plus"],
            ).astype(str)

        if "Position" in out.columns:
            out["PositionBin"] = pd.cut(
                out["Position"],
                bins=[-np.inf, 3, 8, 14, np.inf],
                labels=["front", "upper_mid", "lower_mid", "back"],
            ).astype(str)

        if has(["Cumulative_Degradation", "LapNumber"]):
            out["DegPerRaceLap"] = self.safe_div(out["Cumulative_Degradation"], out["LapNumber"].clip(lower=1), eps)

        if has(["Cumulative_Degradation", "TyreLife"]):
            out["DegPerTyreLap"] = self.safe_div(out["Cumulative_Degradation"], out["TyreLife"].clip(lower=1), eps)

        if "LapTime_Delta" in out.columns:
            out["DeltaAbs"] = out["LapTime_Delta"].abs()

        if "Position_Change" in out.columns:
            out["Abs_Position_Change"] = out["Position_Change"].abs()
            out["Gained_Position"] = (out["Position_Change"] > 0).astype(np.int8)
            out["Lost_Position"] = (out["Position_Change"] < 0).astype(np.int8)

        if has(["Position", "RaceProgress"]):
            out["PositionPressure"] = out["Position"] * out["RaceProgress"]

        self._add_cross(out, "Race_Year", ["Race", "Year"])
        self._add_cross(out, "Compound_Stint", ["Compound", "Stint"])
        self._add_cross(out, "Race_Compound", ["Race", "Compound"])
        self._add_cross(out, "RacePhase_TyreLifeBin", ["RacePhase", "TyreLifeBin"])

        out = out.replace([np.inf, -np.inf], np.nan)
        for col in out.select_dtypes(include=["float64"]).columns:
            out[col] = out[col].astype(np.float32)
        return out

    @staticmethod
    def _add_cross(df, name, cols):
        if set(cols).issubset(df.columns):
            value = df[cols[0]].astype(str)
            for col in cols[1:]:
                value = value + "_" + df[col].astype(str)
            df[name] = value

    def transform_all(self, train_raw, test_raw, original_raw=None):
        train = train_raw.copy()
        test = test_raw.copy()
        original = original_raw.copy() if original_raw is not None and self.cfg.target in original_raw.columns else None

        train["IsOriginalData"] = 0
        test["IsOriginalData"] = 0
        if original is not None:
            original["IsOriginalData"] = 1
            original = original.drop(columns=["Normalized_TyreLife"], errors="ignore")

        train = self.add_features(train)
        test = self.add_features(test)
        original = self.add_features(original) if original is not None else None

        train, test, original = self._align_columns(train, test, original)
        train, test, original = self._fill_missing(train, test, original)
        return train, test, original

    def _align_columns(self, train, test, original):
        exclude_cols = [self.cfg.id_col, self.cfg.target]
        self.feature_cols = [col for col in train.columns if col in test.columns and col not in exclude_cols]
        train = train[self.feature_cols + [self.cfg.target]]
        test = test[self.feature_cols]

        if original is not None:
            for col in self.feature_cols:
                if col not in original.columns:
                    original[col] = np.nan
            original = original[self.feature_cols + [self.cfg.target]]
        return train, test, original

    def _fill_missing(self, train, test, original):
        frames = [train, test]
        if original is not None:
            frames.append(original)

        self.cat_cols = []
        for col in self.feature_cols:
            if any(
                str(frame[col].dtype).startswith(("object", "category", "string"))
                for frame in frames
                if col in frame.columns
            ):
                self.cat_cols.append(col)
        self.num_cols = [col for col in self.feature_cols if col not in self.cat_cols]

        for col in self.cat_cols:
            values = pd.concat([frame[col].astype("string") for frame in frames if col in frame.columns], axis=0)
            mode_value = values.mode().iloc[0] if len(values.mode()) else "__MISSING__"
            for frame in frames:
                if col in frame.columns:
                    frame[col] = frame[col].astype("string").fillna(mode_value).astype(str)

        for col in self.num_cols:
            values = pd.concat([frame[col] for frame in frames if col in frame.columns], axis=0)
            fill_value = values.replace([np.inf, -np.inf], np.nan).median()
            for frame in frames:
                if col in frame.columns:
                    frame[col] = frame[col].replace([np.inf, -np.inf], np.nan).fillna(fill_value)
                    if frame[col].dtype == "float64":
                        frame[col] = frame[col].astype(np.float32)
        return train, test, original

    def feature_report(self, train, test, original):
        rows = [
            {"dataset": "train", "rows": len(train), "columns": train.shape[1], "missing_values": int(train.isna().sum().sum())},
            {"dataset": "test", "rows": len(test), "columns": test.shape[1], "missing_values": int(test.isna().sum().sum())},
        ]
        if original is not None:
            rows.append({"dataset": "original", "rows": len(original), "columns": original.shape[1], "missing_values": int(original.isna().sum().sum())})
        return pd.DataFrame(rows)

    def feature_group_report(self):
        return pd.DataFrame(
            [
                {"group": "all_features", "count": len(self.feature_cols)},
                {"group": "categorical", "count": len(self.cat_cols)},
                {"group": "numeric", "count": len(self.num_cols)},
            ]
        )


feature_engineer = FeatureEngineer(cfg)
train_fe, test_fe, original_fe = feature_engineer.transform_all(train_raw, test_raw, original_raw)

display(feature_engineer.feature_report(train_fe, test_fe, original_fe))
display(feature_engineer.feature_group_report())
display(pd.DataFrame({"categorical_feature": feature_engineer.cat_cols[:15]}))


## Matrix Preparation

In [ ]:
"""Prepare model matrices and keep the same feature order for validation and test prediction."""

class MatrixBuilder:
    def __init__(self, cfg: Config, feature_engineer: FeatureEngineer):
        self.cfg = cfg
        self.feature_engineer = feature_engineer

    def build(self, train, test, original=None):
        X_comp = train.drop(columns=[self.cfg.target], errors="ignore")
        y_comp = train[self.cfg.target].astype(int)
        X_test = test.copy()

        if original is not None and self.cfg.target in original.columns:
            X_orig = original.drop(columns=[self.cfg.target], errors="ignore")
            y_orig = original[self.cfg.target].astype(int)
        else:
            X_orig, y_orig = None, None

        common_features = [col for col in X_comp.columns if col in X_test.columns]
        X_comp = X_comp[common_features]
        X_test = X_test[common_features]

        if X_orig is not None:
            for col in common_features:
                if col not in X_orig.columns:
                    X_orig[col] = np.nan
            X_orig = X_orig[common_features]

        cat_cols = [col for col in self.feature_engineer.cat_cols if col in common_features]
        cat_features_idx = [X_comp.columns.get_loc(col) for col in cat_cols]
        return X_comp, y_comp, X_test, X_orig, y_orig, cat_cols, cat_features_idx

    def matrix_report(self, X_comp, y_comp, X_test, X_orig, y_orig, cat_cols):
        rows = [
            {"matrix": "X_comp", "rows": len(X_comp), "columns": X_comp.shape[1], "target_rate": y_comp.mean()},
            {"matrix": "X_test", "rows": len(X_test), "columns": X_test.shape[1], "target_rate": np.nan},
        ]
        if X_orig is not None and y_orig is not None:
            rows.append({"matrix": "X_orig", "rows": len(X_orig), "columns": X_orig.shape[1], "target_rate": y_orig.mean()})
        rows.append({"matrix": "categorical_features", "rows": len(cat_cols), "columns": np.nan, "target_rate": np.nan})
        return pd.DataFrame(rows)


matrix_builder = MatrixBuilder(cfg, feature_engineer)
X_comp, y_comp, X_test, X_orig, y_orig, cat_cols, cat_features_idx = matrix_builder.build(train_fe, test_fe, original_fe)

display(matrix_builder.matrix_report(X_comp, y_comp, X_test, X_orig, y_orig, cat_cols))


## CatBoost Validation

In [ ]:
"""Train CatBoost with grouped folds and report only the key metrics in simple tables."""

from sklearn.metrics import f1_score, log_loss, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold
from catboost import CatBoostClassifier


class CatBoostPitStopModel:
    def __init__(self, cfg: Config, cat_features_idx: list[int]):
        self.cfg = cfg
        self.cat_features_idx = cat_features_idx
        self.best_iters: list[int] = []
        self.fold_metrics: list[dict] = []
        self.oof_pred = None
        self.final_model = None
        self.fold_importances: list[np.ndarray] = []

    def params(self, seed, iterations, validation=True):
        params = {
            "iterations": iterations,
            "learning_rate": 0.018,
            "depth": 8,
            "l2_leaf_reg": 8.5,
            "random_strength": 0.65,
            "bootstrap_type": "Bayesian",
            "bagging_temperature": 0.45,
            "loss_function": "Logloss",
            "eval_metric": "AUC",
            "auto_class_weights": "Balanced",
            "task_type": "GPU",
            "devices": "0:1",
            "random_seed": seed,
            "allow_writing_files": False,
            "verbose": 300,
        }
        if validation:
            params["early_stopping_rounds"] = 500
        return params

    @staticmethod
    def best_threshold(y_true, pred):
        thresholds = np.linspace(0.05, 0.95, 181)
        scores = [f1_score(y_true, pred >= threshold) for threshold in thresholds]
        best_idx = int(np.argmax(scores))
        return float(thresholds[best_idx]), float(scores[best_idx])

    def validate(self, X_comp, y_comp, X_orig=None, y_orig=None, groups=None):
        if groups is None:
            groups = np.arange(len(X_comp))

        splitter = StratifiedGroupKFold(n_splits=self.cfg.n_folds, shuffle=True, random_state=self.cfg.seed)
        self.oof_pred = np.zeros(len(X_comp), dtype=float)
        self.best_iters = []
        self.fold_metrics = []
        self.fold_importances = []

        for fold, (tr_idx, val_idx) in enumerate(splitter.split(X_comp, y_comp, groups=groups), 1):
            X_tr_comp = X_comp.iloc[tr_idx].reset_index(drop=True)
            y_tr_comp = y_comp.iloc[tr_idx].reset_index(drop=True)
            X_val = X_comp.iloc[val_idx].reset_index(drop=True)
            y_val = y_comp.iloc[val_idx].reset_index(drop=True)

            if X_orig is not None and y_orig is not None:
                X_tr = pd.concat([X_tr_comp, X_orig.reset_index(drop=True)], axis=0, ignore_index=True)
                y_tr = pd.concat([y_tr_comp, y_orig.reset_index(drop=True)], axis=0, ignore_index=True)
            else:
                X_tr = X_tr_comp.copy()
                y_tr = y_tr_comp.copy()

            model = CatBoostClassifier(**self.params(seed=self.cfg.seed + fold, iterations=11000, validation=True))
            model.fit(
                X_tr,
                y_tr,
                eval_set=(X_val, y_val),
                cat_features=self.cat_features_idx,
                use_best_model=True,
            )

            val_pred = np.clip(model.predict_proba(X_val)[:, 1], 1e-7, 1 - 1e-7)
            self.oof_pred[val_idx] = val_pred
            best_thr, best_f1 = self.best_threshold(y_val, val_pred)
            y_hat_05 = val_pred >= 0.5
            y_hat_best = val_pred >= best_thr
            best_iter = model.get_best_iteration()
            self.best_iters.append(best_iter)
            self.fold_importances.append(model.get_feature_importance())
            self.fold_metrics.append(
                {
                    "fold": fold,
                    "train_rows": len(X_tr),
                    "valid_rows": len(X_val),
                    "auc": roc_auc_score(y_val, val_pred),
                    "logloss": log_loss(y_val, val_pred),
                    "f1_0_5": f1_score(y_val, y_hat_05),
                    "best_threshold": best_thr,
                    "f1_best": best_f1,
                    "precision_best": precision_score(y_val, y_hat_best),
                    "recall_best": recall_score(y_val, y_hat_best),
                    "best_iter": best_iter,
                }
            )
            gc.collect()

        return pd.DataFrame(self.fold_metrics)

    def validation_feature_importance(self, feature_names):
        if not self.fold_importances:
            return pd.DataFrame(columns=["feature", "importance"])
        importance = np.mean(np.vstack(self.fold_importances), axis=0)
        return pd.DataFrame({"feature": feature_names, "importance": importance}).sort_values("importance", ascending=False)

    def overall_report(self, y_true):
        best_thr, best_f1 = self.best_threshold(y_true, self.oof_pred)
        y_hat_best = self.oof_pred >= best_thr
        return pd.DataFrame(
            [
                {
                    "auc": roc_auc_score(y_true, self.oof_pred),
                    "logloss": log_loss(y_true, self.oof_pred),
                    "best_threshold": best_thr,
                    "f1_best": best_f1,
                    "precision_best": precision_score(y_true, y_hat_best),
                    "recall_best": recall_score(y_true, y_hat_best),
                    "mean_best_iter": np.mean(self.best_iters),
                }
            ]
        )

    @staticmethod
    def prediction_summary(pred):
        return pd.DataFrame(
            [
                {
                    "min": pred.min(),
                    "p01": np.percentile(pred, 1),
                    "p05": np.percentile(pred, 5),
                    "p25": np.percentile(pred, 25),
                    "median": np.median(pred),
                    "p75": np.percentile(pred, 75),
                    "p95": np.percentile(pred, 95),
                    "p99": np.percentile(pred, 99),
                    "max": pred.max(),
                    "mean": pred.mean(),
                }
            ]
        )

    @staticmethod
    def feature_importance(model, feature_names):
        return pd.DataFrame(
            {
                "feature": feature_names,
                "importance": model.get_feature_importance(),
            }
        ).sort_values("importance", ascending=False)

    def fit_final(self, X_comp, y_comp, X_orig=None, y_orig=None):
        if not self.best_iters:
            raise RuntimeError("Run validation before fitting the final model.")

        final_iterations = max(1800, int(np.mean(self.best_iters) * 1.10))
        if X_orig is not None and y_orig is not None:
            X_full = pd.concat([X_comp.reset_index(drop=True), X_orig.reset_index(drop=True)], axis=0, ignore_index=True)
            y_full = pd.concat([y_comp.reset_index(drop=True), y_orig.reset_index(drop=True)], axis=0, ignore_index=True)
        else:
            X_full = X_comp.copy()
            y_full = y_comp.copy()

        self.final_model = CatBoostClassifier(
            **self.params(seed=self.cfg.seed + 999, iterations=final_iterations, validation=False)
        )
        self.final_model.fit(X_full, y_full, cat_features=self.cat_features_idx)
        final_stats = pd.DataFrame(
            [
                {
                    "train_rows": len(X_full),
                    "test_features": X_full.shape[1],
                    "final_iterations": final_iterations,
                    "target_rate": y_full.mean(),
                }
            ]
        )
        return self.final_model, X_full, final_stats


In [ ]:
"""Run validation and show compact fold metrics, overall metrics, prediction spread, and feature importance."""

trainer = CatBoostPitStopModel(cfg, cat_features_idx)
groups = train_raw["Race"].astype(str) + "_" + train_raw["Year"].astype(str)

fold_metrics = trainer.validate(X_comp, y_comp, X_orig=X_orig, y_orig=y_orig, groups=groups)
overall_metrics = trainer.overall_report(y_comp)
validation_prediction_summary = trainer.prediction_summary(trainer.oof_pred)
validation_feature_importance = trainer.validation_feature_importance(X_comp.columns)

display(fold_metrics.round(6))
display(overall_metrics.round(6))
display(validation_prediction_summary.round(6))
display(validation_feature_importance.head(25).round(6))


## Final Model and Files

In [ ]:
"""Fit the final model on all available labeled data, create predictions, and save files."""

final_model, X_full, final_training_stats = trainer.fit_final(X_comp, y_comp, X_orig=X_orig, y_orig=y_orig)
test_pred = np.clip(final_model.predict_proba(X_test)[:, 1], 1e-7, 1 - 1e-7)

final_feature_importance = trainer.feature_importance(final_model, X_full.columns)
test_prediction_summary = trainer.prediction_summary(test_pred)

submission = sample_submission.copy()
target_col = cfg.target if cfg.target in submission.columns else [col for col in submission.columns if col != cfg.id_col][0]
submission[target_col] = test_pred
submission.to_csv("submission.csv", index=False)

pd.DataFrame({"y_true": y_comp.values, "oof_pred": trainer.oof_pred}).to_csv("oof_predictions.csv", index=False)
final_feature_importance.to_csv("final_feature_importance.csv", index=False)

saved_files = pd.DataFrame(
    [
        {"file": "submission.csv", "rows": len(submission)},
        {"file": "oof_predictions.csv", "rows": len(y_comp)},
        {"file": "final_feature_importance.csv", "rows": len(final_feature_importance)},
    ]
)

display(final_training_stats.round(6))
display(test_prediction_summary.round(6))
display(final_feature_importance.head(25).round(6))
display(saved_files)
